# Daten laden und erste Exploration

In diesem Notebook wird der Datensatz geladen und zunächst untersucht. Ziel ist es, einen Überblick über die Struktur der Daten, die vorhandenen Spalten sowie mögliche fehlende Werte zu erhalten.

In [17]:
import pandas as pd

## Datensatz laden

Der Datensatz wird mit der Bibliothek pandas in die Python-Umgebung geladen. Anschließend werden die ersten Zeilen angezeigt, um einen ersten Eindruck der Daten zu erhalten.

In [18]:
df = pd.read_csv("../data/consumer_complaints.csv")

In [19]:
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2020-07-06,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,346XX,NaN,Other,Web,2020-07-06,Closed with explanation,Yes,NaN,3730948
1,2019-12-26,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,NaN,NaN,CAPITAL ONE FINANCIAL CORPORATION,CA,94025,NaN,Consent not provided,Web,2019-12-26,Closed with explanation,Yes,NaN,3477549
2,2020-05-08,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,These are not my accounts.,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NV,89030,NaN,Consent provided,Web,2020-05-08,Closed with explanation,Yes,NaN,3642453
3,2024-01-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,Kindly address this issue on my credit report....,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,60502,NaN,Consent provided,Web,2024-01-05,Closed with non-monetary relief,Yes,NaN,8113747
4,2024-01-21,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,NC,27401,Servicemember,Consent not provided,Web,2024-01-21,Closed with explanation,Yes,NaN,8191825


## Überblick über die Datenstruktur

Mit `df.info()` wird untersucht:
- wie viele Zeilen und Spalten vorhanden sind,
- welche Datentypen verwendet werden,
- und ob fehlende Werte existieren.

Dies ist wichtig, um die Datenqualität vor der Analyse zu prüfen.

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15392946 entries, 0 to 15392945
Data columns (total 18 columns):
 #   Column                        Dtype
---  ------                        -----
 0   Date received                 str  
 1   Product                       str  
 2   Sub-product                   str  
 3   Issue                         str  
 4   Sub-issue                     str  
 5   Consumer complaint narrative  str  
 6   Company public response       str  
 7   Company                       str  
 8   State                         str  
 9   ZIP code                      str  
 10  Tags                          str  
 11  Consumer consent provided?    str  
 12  Submitted via                 str  
 13  Date sent to company          str  
 14  Company response to consumer  str  
 15  Timely response?              str  
 16  Consumer disputed?            str  
 17  Complaint ID                  int64
dtypes: int64(1), str(17)
memory usage: 2.1 GB


## Analyse fehlender Werte

In diesem Schritt wird überprüft, welche Spalten fehlende Werte enthalten. Fehlende Werte können die spätere Analyse beeinflussen und müssen daher identifiziert werden.

In [21]:
df.isnull().sum()

Date received                          0
Product                                0
Sub-product                       235276
Issue                                  6
Sub-issue                         908664
Consumer complaint narrative    11606511
Company public response          7076903
Company                                0
State                              61408
ZIP code                           30265
Tags                            14638798
Consumer consent provided?       2507002
Submitted via                          0
Date sent to company                   0
Company response to consumer          21
Timely response?                       0
Consumer disputed?              14625061
Complaint ID                           0
dtype: int64

## Untersuchung der vorhandenen Spalten

Zur Vorbereitung der NLP-Analyse werden alle Spaltennamen angezeigt. Ziel ist es, diejenige Spalte zu identifizieren, die die eigentlichen Beschwerdetexte enthält.

In [22]:
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')

## Identifikation der relevanten Textspalte

Die Spalte `Consumer complaint narrative` enthält die eigentlichen freien Beschwerdetexte der Verbraucher:innen. Dies lässt sich daran erkennen, dass die Inhalte vollständige Sätze und längere Beschreibungen von Problemen enthalten.

Diese Spalte ist für die NLP-Analyse besonders wichtig, da:
- sie unstrukturierte Textdaten enthält,
- die Beschwerden in natürlicher Sprache formuliert sind,
- und daraus Themen, Muster und häufige Probleme extrahiert werden können.

Die weiteren Schritte der Textanalyse basieren daher hauptsächlich auf dieser Spalte.

In [23]:
df['Consumer complaint narrative'].head()

0                                                  NaN
1                                                  NaN
2                           These are not my accounts.
3    Kindly address this issue on my credit report....
4                                                  NaN
Name: Consumer complaint narrative, dtype: str

## Erste Beobachtungen

Die ersten Beispiele zeigen, dass die Beschwerden:
- unterschiedlich lang sind,
- viele Sonderzeichen und Formatierungen enthalten,
- und sprachlich nicht standardisiert sind.

Daher ist eine Vorverarbeitung notwendig, bevor die Texte mit NLP-Methoden analysiert werden können.

## Entfernen fehlender Beschwerdetexte

Einige Einträge enthalten keine eigentlichen Beschwerdetexte und werden als `NaN` angezeigt. Diese Datensätze können nicht für die NLP-Analyse verwendet werden und müssen daher entfernt werden.

In [24]:
df = df.dropna(subset=['Consumer complaint narrative'])

In [25]:
df['Consumer complaint narrative'].isnull().sum()

np.int64(0)

## Vereinheitlichung der Spaltennamen

Zur Vereinfachung der weiteren Analyse werden die Spaltennamen vereinheitlicht. Dabei werden Leerzeichen durch Unterstriche ersetzt und alle Spaltennamen in Kleinbuchstaben umgewandelt.

In [26]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [27]:
df.columns

Index(['date_received', 'product', 'sub-product', 'issue', 'sub-issue',
       'consumer_complaint_narrative', 'company_public_response', 'company',
       'state', 'zip_code', 'tags', 'consumer_consent_provided?',
       'submitted_via', 'date_sent_to_company', 'company_response_to_consumer',
       'timely_response?', 'consumer_disputed?', 'complaint_id'],
      dtype='str')

In [28]:
df['consumer_complaint_narrative']

2                                  These are not my accounts.
3           Kindly address this issue on my credit report....
5           I wrote three requests, the unverified account...
6           XXXX XXXX has a old account settled in XXXX th...
7           They call at all hours and on the weekends usi...
                                  ...                        
15392922    All three credit reporting agencies are report...
15392925    My credit was impacted negatively from XXXX XX...
15392926    These inquires on my report weren't initiated ...
15392927    On XX/XX/XXXX, I observed an authorized severa...
15392945    The consumer reporting agency, XXXX, has furni...
Name: consumer_complaint_narrative, Length: 3786435, dtype: str

## Zusammenfassung der ersten Datenanalyse

Der Datensatz wurde erfolgreich geladen und untersucht. Die Spalte `consumer_complaint_narrative` wurde als wichtigste Textquelle für die NLP-Analyse identifiziert. Zusätzlich wurden fehlende Werte entfernt und die Spaltennamen vereinheitlicht, um die weitere Verarbeitung zu erleichtern.

Damit sind die Daten für die Vorverarbeitung vorbereitet.